In [1]:
from core.analysis_config import AnalysisConfig
from core.reference import Reference
from core.observable import get_obs_info, ObsType
from utils.analyzer import Analyzer
from utils.plot_config import PlotLimits, CMSPlotStyle
from utils.workdirectory import WorkDirectory
from pathlib import Path
CONFIG = AnalysisConfig('bamboo_hh/config/disc_study_new.yml')
plot_style = CMSPlotStyle(figsize=(8,6), label_fs=18, tick_fs=18, legend_fs=18, cms_fs=18)

Welcome to JupyROOT 6.30/02


In [ ]:
from utils.plot_config import PlotLimits, CMSPlotStyle
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import mplhep
plt.style.use(mplhep.style.CMS)

from utils.plots import hist_to_numpy, format_label_for_plt, save_fig, hist_to_numpy

def plot_log_sig_bkg(analyzer, ref, plot_style = CMSPlotStyle(), plot_limits: PlotLimits = None, eras=None):
    signal_hist, background_hist = analyzer.get_signal_background(ref, eras=eras)
    plot_style, plot_limits = analyzer._build_ax_labels_limits(ref, plot_style, plot_limits=plot_limits)
    save_path = analyzer.outdir / f"{ref.name}_log.pdf"
    
    hists = [signal_hist, background_hist]
    legend = ['Signal', 'Background']
    colors = ['blue', 'red']
    
    # if ObsType.get_dimensionality(ref) == 1:
    hist_values, hist_edges = [], []
    for hist in hists:
        vals, edges = hist_to_numpy(hist)
        hist_values.append(vals)
        hist_edges.append(edges[0])
    
    plot_style.ylabel = 'Events'
    # Auto-scale y if not specified
    # plot_limits.ymin = plot_limits.ymin or 0
    # plot_limits.ymax = plot_limits.ymax or max(vals.max() for vals in hist_values) * 1.2
        
    values = hist_values
    edges = hist_edges
    fig, ax = plot_style.setup_figure()
    for i in range(len(values)):
        values_i = values[i]
        color_i = colors[i]
        legend_i = legend[i]
        edges_i = edges[i]
        ax.step(edges_i[:-1], values_i, where='post', color=color_i, linewidth=2.0, label=legend_i)
        ax.fill_between(edges_i[:-1], values_i, step='post', color=color_i, alpha=0.2)

    xlabel = format_label_for_plt(plot_style.xlabel)
    ylabel = format_label_for_plt(plot_style.ylabel)
    print(xlabel)
    ax.set_xlabel(xlabel, fontsize=plot_style.label_fs)
    ax.set_ylabel(ylabel, fontsize=plot_style.label_fs)
    ax.tick_params(axis='both', which='major', labelsize=plot_style.tick_fs)
    # ax.set_xlim(plot_limits.xmin, plot_limits.xmax)
    # ax.set_ylim(plot_limits.ymin, plot_limits.ymax)
    
    # ax.set_yscale('log')

    ax.grid(True)
    ax.legend(fontsize=plot_style.legend_fs, loc="upper right", frameon=True)

    plot_style.add_cms_text(ax)
    fig.tight_layout()
    save_fig(plt, save_path)
    plt.show()

In [ ]:
wd = WorkDirectory('/eos/user/a/anunezde/Z_OUTPUT_eos/Triggers_New/2022_JetTop_pt10')
analyzer = Analyzer(wd, CONFIG)
analyzer.plot_sig_bkg(Reference('SL_e_resolved'), plot_style, PlotLimits(xmin=-6, xmax=8, ymax=0.06))